# PyMMM Pipeline: ND2 → Zarr

This notebook demonstrates the full PyMMM pipeline:
1. Load an ND2 file lazily
2. Register (drift-correct) the data
3. Detect feeding lanes
4. Detect trenches within lanes
5. Extract trenches to a compressed zarr store

Each stage checkpoints to a companion zarr store, so you can restart the kernel and resume from any checkpoint.

In [1]:
## enable jupyter autoreload magics
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import sys
import os
sys.path.insert(0, "..")

In [3]:
from pymmm import ND2Experiment, Registrator, LaneDetector, TrenchDetector, Extractor
from pymmm.checkpoint import CompanionStore
from dask.distributed import Client, LocalCluster
import hvplot.xarray
import hvplot

hvplot.extension('bokeh')

# Start a dask cluster for parallelisation
cluster = LocalCluster(processes=True, threads_per_worker=1, n_workers=0, memory_limit="10GB")
client = Client(cluster)
client

/home/georgeos/GitHub/PyMMM/.pixi/envs/default/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33605 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:33605/status,
Dashboard: http://127.0.0.1:33605/status,Workers: 0
Total threads: 0,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41113,Workers: 0
Dashboard: http://127.0.0.1:33605/status,Total threads: 0
Started: Just now,Total memory: 0 B


## 1. Load ND2 file

In [4]:
exp = ND2Experiment("/data/20260331_SB5_6_7_8_16/20260331.nd2") 
exp#.select_fovs(["XYPos:10","XYPos:18"]).select_times(0,200)

# Optional: crop to a spatial ROI
#exp.select_roi(y=(570, 1000), x=(0, -1))

print(exp)
cluster.adapt(minimum=0, maximum=1)
# Interactive browse raw data
exp.data.hvplot.image(
    x="X", y="Y", cmap="Greys_r", dynamic=True,
    rasterize=True, widget_location="top", aspect="equal"
)

ND2Experiment: 20260331
  Path: /data/20260331_SB5_6_7_8_16/20260331.nd2
  Dims: T=8 × P=25 × C=4 × Y=2304 × X=2304
  FOVs: 25  (XYPos:0, XYPos:1, XYPos:2...)
  Channels: PC, CFP, mVenus, mCherry
  Timepoints: 8
  Pixel size: 0.1079 µm
  Time interval: 299992.7 ms


BokehModel(combine_events=True, render_bundle={'docs_json': {'11dc3b11-0391-4b7c-90e7-81a053531354': {'version…

In [5]:
# Optional: discard unwanted FOVs
# exp.discard_fovs(["xy029", "xy030"])

## 2. Companion store

In [6]:
store = CompanionStore.for_experiment(exp)
store

CompanionStore(/data/20260307_SB7_exit_snake_V4_1.pymmm.zarr, [registration, lane_detection, trench_detection])

## 3. Registration (Checkpoint 1)

Register the experiment to correct for stage drift.

In [7]:
cluster.adapt(minimum=0, maximum=32)
#store.reset("registration")
if store.has_registration():
    print("Loading registration from checkpoint...")
    reg = Registrator.load(exp, store)
else:
    reg = Registrator(
        exp, store,
        registration_channel="PC",  # ← Change to your phase-contrast channel
        mode="first",
        rotation=0,               # ← Adjust rotation if needed
        roi={"y": (350, 1400), "x": (750, 1750)},
    )
    reg.compute_mean_images(plot=False)
    reg.compute_tmats(plot=True)
    reg.save()

Loading registration from checkpoint...


In [8]:
# Interactive browse registered data
cluster.adapt(minimum=0, maximum=1)
reg.get_stabilized_data().hvplot.image(
    x="X", y="Y", cmap="Greys_r", dynamic=True,
    rasterize=True, widget_location="top", aspect="equal"
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'99f29f52-489e-42a8-abfa-274ec8c9e12e': {'version…

In [9]:
# Drift diagnostics for a specific FOV
#reg.plot_drift(fov=2)

## 4. Lane Detection (Checkpoint 2)

Detect the feeding lane y-positions in each FOV.

In [10]:
#store.reset("lane_detection")
if store.has_lane_detection():
    print("Loading lane detection from checkpoint...")
    lane_det = LaneDetector.load(exp, reg, store)
else:
    lane_det = LaneDetector(exp, reg, store, detection_channel="PC")

Loading lane detection from checkpoint...


In [11]:
# Tune sliders, browse FOVs, then click "Apply to all FOVs"
#lane_det.interactive_detect_lanes()

In [12]:
#for fov, lanes in lane_det._lanes.items():
#    for lane in lanes:
#        lane.orientation = 1  

# Then re-save
#lane_det.save()

In [13]:
# After tuning sliders and clicking "Apply to all FOVs", save the results
#lane_det.save()

In [14]:
# Inspect another FOV
# lane_det.plot_fov(fov=1)

## 5. Trench Detection (Checkpoint 3)

Detect trench x-positions within each lane.

In [15]:
#store.reset("trench_detection")
if store.has_trench_detection():
    print("Loading trench detection from checkpoint...")
    trench_det = TrenchDetector.load(exp, reg, lane_det, store)
else:
    trench_det = TrenchDetector(exp, reg, lane_det, store, detection_channel="PC")

Loading trench detection from checkpoint...


In [16]:
# Tune sliders, browse FOVs, then click "Apply to all FOVs"
#trench_det.interactive_detect_trenches()

In [17]:
# After tuning sliders and clicking "Apply to all FOVs", save the results
#trench_det.save()
#trench_det.get_trench_table()

In [18]:
#y_bottom - y_top for the first row of trench_det.get_trench_table()
#trench_det.get_trench_table().iloc[0]['y_bottom'] - trench_det.get_trench_table().iloc[0]['y_top']

In [19]:
# # Preview a single trench before full extraction
# from pymmm.plotting import plot_trench_preview
# preview = trench_det_extractor_preview = Extractor(exp, reg, trench_det)
# single = preview.extract_single_trench(trench_id=0)
# plot_trench_preview(single, trench_id=0, n_frames=8)

## 6. Extraction

Extract all trenches to a compressed zarr store.

In [20]:
cluster.adapt(minimum=32, maximum=32)

In [21]:
extractor = Extractor(exp, reg, trench_det, output_path="/data/20260307_SB7_exit_snake_V4_1_with_metadata.trenches.zarr")
print(extractor)  # dimensions, chunks, size estimate
extractor.preview()

Extractor: 20260307_SB7_exit_snake_V4_1
  Output: /data/20260307_SB7_exit_snake_V4_1_with_metadata.trenches.zarr
  Shape: Trench=2795 × T=721 × C=2 × Y=164 × X=34
  Chunks: (1, 721, 1, 164, 34)
  Dtype: uint16
  Size (uncompressed): 41.9 GB
  Trenches: 2795 across 28 FOVs
  Trench size: 164 × 34 px (17.7 × 3.7 µm)
  Channels: PC, mCherry
  Timepoints: 721


<xarray.DataArray 'concatenate-ef68a4b9ce0175846c60bac772afe3d5' (Trench: 2795,
                                                                  T: 721, C: 2,
                                                                  Y: 164, X: 34)> Size: 45GB
dask.array<concatenate, shape=(2795, 721, 2, 164, 34), dtype=uint16, chunksize=(1, 721, 1, 164, 34), chunktype=numpy.ndarray>
Coordinates:
  * Trench   (Trench) int32 11kB 0 1 2 3 4 5 6 ... 2789 2790 2791 2792 2793 2794
  * C        (C) <U7 56B 'PC' 'mCherry'
  * Y        (Y) int64 1kB 0 1 2 3 4 5 6 7 8 ... 156 157 158 159 160 161 162 163
  * X        (X) int64 272B 0 1 2 3 4 5 6 7 8 9 ... 25 26 27 28 29 30 31 32 33
Dimensions without coordinates: T

In [22]:
extractor.extract(compressor='zstd', clevel=3, show_progress=True)

Extracting FOVs:   0%|          | 0/28 [00:00<?, ?it/s]

Extraction complete: /data/20260307_SB7_exit_snake_V4_1_with_metadata.trenches.zarr
  Shape: (2795, 721, 2, 164, 34), Chunks: (1, 721, 1, 164, 34)


## 7. Verify output

In [24]:
import zarr

z = zarr.open(str(extractor.output_path), mode='r')
print("Shape:", z['data'].shape)
print("Chunks:", z['data'].chunks)
print("Attrs:", dict(z.attrs))

Shape: (2795, 721, 2, 164, 34)
Chunks: (1, 721, 1, 164, 34)
Attrs: {'extractor_store_version': 1, 'layout': 'xarray_trench_store_v1', 'source_metadata_version': 2, 'source_nd2': '/data/20260307_SB7_exit_snake_V4_1.nd2', 'experiment_name': '20260307_SB7_exit_snake_V4_1', 'pixel_size_um': 0.107869821220548, 'registration_params': {'channel': 'PC', 'mode': 'first', 'rotation': 0}, 'source_subset_metadata': {'raw_fov_names': ['XYPos:0', 'XYPos:1', 'XYPos:2', 'XYPos:3', 'XYPos:4', 'XYPos:5', 'XYPos:6', 'XYPos:7', 'XYPos:8', 'XYPos:9', 'XYPos:10', 'XYPos:11', 'XYPos:12', 'XYPos:13', 'XYPos:14', 'XYPos:15', 'XYPos:16', 'XYPos:17', 'XYPos:18', 'XYPos:19', 'XYPos:20', 'XYPos:21', 'XYPos:22', 'XYPos:23', 'XYPos:24', 'XYPos:25', 'XYPos:26', 'XYPos:27'], 'active_fov_names': ['XYPos:0', 'XYPos:1', 'XYPos:2', 'XYPos:3', 'XYPos:4', 'XYPos:5', 'XYPos:6', 'XYPos:7', 'XYPos:8', 'XYPos:9', 'XYPos:10', 'XYPos:11', 'XYPos:12', 'XYPos:13', 'XYPos:14', 'XYPos:15', 'XYPos:16', 'XYPos:17', 'XYPos:18', 'XYPos:1